# C01: データ準備（最新HTMLの差分ダウンロード）

**最終更新日**: 2026-04-29

| 改訂日 | 内容 |
|--------|------|
| 2026-04-29 | ダウンロードskip判定を「Parquet含有」→「ディスク上HTMLの有無」に変更（race_id ≠ 日付の前提に対応。Parquetにあっても HTML が無いレースを取得できるようにした） |

---

このノートブックでは以下の処理を行います：
1. 最新レースHTMLの一括ダウンロード
2. データ確認

**使い方**: 上から順番にセルを実行してください（`Shift + Enter`）


In [ ]:
# == Colab用: GitHubからリポジトリをクローンして準備 ==
!git clone https://github.com/iinumac/keiba_prediction.git
%cd keiba_prediction

# クローンしたディレクトリがPROJECT_ROOTになる
import os
os.environ['PROJECT_ROOT'] = '/content/keiba_prediction'
print("✅ リポジトリのクローン完了。現在のディレクトリ:", os.getcwd())

Cloning into 'keiba_prediction'...
remote: Enumerating objects: 55546, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 55546 (delta 2), reused 9 (delta 1), pack-reused 55533 (from 2)
Receiving objects: 100% (55546/55546), 197.68 MiB | 26.41 MiB/s, done.
Resolving deltas: 100% (55213/55213), done.
Updating files: 100% (55357/55357), done.
/content/keiba_prediction
✅ リポジトリのクローン完了。現在のディレクトリ: /content/keiba_prediction


---
## ステップ1: 環境設定

In [ ]:
import sys
import os
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
from datetime import datetime
import time
import requests
from pathlib import Path

# プロジェクトルート（notebooks/sagemaker/ から2階層上）
PROJECT_ROOT = Path('.').resolve()  # Colab環境ではカレントディレクトリがルート
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

# データパス
HTML_DIR = PROJECT_ROOT / 'data' / 'raceHTML'
HTML_DIR.mkdir(parents=True, exist_ok=True)

print("✅ 環境設定完了")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"HTML_DIR: {HTML_DIR}")

✅ 環境設定完了
PROJECT_ROOT: /content/keiba_prediction
HTML_DIR: /content/keiba_prediction/data/raceHTML


---
## ステップ2: 最新レースHTMLの一括ダウンロード

netkeibaから最新のレースHTMLを自動的にダウンロードします。
- 既に存在するHTMLはスキップ
- レースが存在しない場合は次へスキップ
- 当年のみ

In [ ]:
def generate_race_ids(start_year=2010, end_year=None, debug=False):
    """
    レースIDを生成するジェネレータ

    race_id形式: YYYYCCRRDDNN (12桁)
        YYYY: 年 (2010-現在)
        CC: 競馬場コード (01-10)
        RR: 開催回 (01-10)
        DD: 開催日 (01-20)
        NN: レース番号 (01-12)
    """
    if end_year is None:
        end_year = datetime.now().year

    for year in range(start_year, end_year + 1):
        for venue_code in range(1, 11):  # 01-10
            kaisai = 1
            while kaisai <= 10:  # 01-10（最大10回）
                day = 1
                kaisai_has_races = False
                skip_kaisai_flag = False

                while day <= 20:  # 01-20（最大20日）
                    race_num = 1

                    while race_num <= 12:  # 01-12
                        race_id = f"{year:04d}{venue_code:02d}{kaisai:02d}{day:02d}{race_num:02d}"
                        skip_signal = yield race_id

                        if skip_signal == 'race_found':
                            kaisai_has_races = True
                        elif skip_signal == 'skip_day':
                            break
                        elif skip_signal == 'skip_kaisai':
                            skip_kaisai_flag = True
                            break

                        race_num += 1

                    if skip_kaisai_flag:
                        break
                    day += 1

                if skip_kaisai_flag:
                    kaisai += 1
                    continue

                if not kaisai_has_races:
                    break

                kaisai += 1

def download_race_html(race_id, force=False):
    """
    レースHTMLをダウンロード
    """
    year = race_id[:4]
    output_path = HTML_DIR / year / f"{race_id}.html"

    if output_path.exists() and not force:
        return 'exists'

    output_path.parent.mkdir(parents=True, exist_ok=True)

    url = f"https://db.netkeiba.com/race/{race_id}"
    headers = {
        'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    }

    try:
        response = requests.get(url, headers=headers, timeout=10)

        if response.status_code != 200:
            return 'not_found'

        response.encoding = 'EUC-JP'
        html_text = response.text

        if len(html_text) < 5000:
            return 'not_found'

        if 'データが存在しません' in html_text or 'お探しのページが見つかりませんでした' in html_text:
            return 'not_found'

        horse_link_count = html_text.count('/horse/')
        if horse_link_count < 3:
            return 'not_found'

        with open(output_path, 'w', encoding='utf-8') as f:
            f.write(html_text)

        return 'success'

    except requests.exceptions.Timeout:
        return 'error'
    except Exception as e:
        return 'error'

print("✅ ダウンロード関数準備完了")

✅ ダウンロード関数準備完了


In [ ]:
# HTMLダウンロードの実行
print("📥 レースHTMLのダウンロードを開始します")

# ========================================================
# skip判定はディスク上のHTML有無のみで行う（download_race_html内で実施）
# 注: race_id は年・日付を表すIDではないため、Parquet の year 列での比較は
#     使用しない。Parquetのロードは「参考表示」用途のみ。
#     Parquetにあっても HTML がローカルに無いケース（前回診断で546件確認）の
#     取りこぼしを防ぐため、Parquet先行skipは廃止。
# ========================================================
from utils.data_loader import load_races
print("📥 既存Parquet（参考情報）を確認中...")
try:
    existing_races_df = load_races(from_github=True)
    print(f"📊 Parquet上のレース数（参考）: {len(existing_races_df):,}件")
except Exception as e:
    print(f"⚠️ Parquetの参照に失敗（無視して続行）: {e}")

print("⚠️ この処理には時間がかかる場合があります\n")

# ダウンロード開始年の決定（当年のみ）
current_year = datetime.now().year

start_year = current_year
print(f"当年のみ更新: {start_year}年～現在\n")

# ダウンロード実行
generator = generate_race_ids(start_year=start_year)
stats = {'success': 0, 'exists': 0, 'not_found': 0, 'error': 0}
last_success_race = None
total_processed = 0

try:
    race_id = next(generator)

    while True:
        # ディスク上HTMLの有無で判定。あれば 'exists', 無ければ HTTP 取得。
        result = download_race_html(race_id)
        stats[result] += 1
        total_processed += 1

        skip_signal = None

        if result == 'success' or result == 'exists':
            last_success_race = race_id
            skip_signal = 'race_found'
            if result == 'success':
                print(f"✅ {race_id}: ダウンロード成功")
                time.sleep(0.5)
        elif result == 'not_found':
            race_num = int(race_id[-2:])
            if race_num == 1:
                day = int(race_id[-4:-2])
                if day == 1:
                    skip_signal = 'skip_kaisai'
                else:
                    skip_signal = 'skip_day'

        if total_processed % 100 == 0:
            print(f"\n📊 進捗: {total_processed}件処理")
            print(f"   成功: {stats['success']}, 既存: {stats['exists']}")
            print(f"   未発見: {stats['not_found']}, エラー: {stats['error']}\n")

        race_id = generator.send(skip_signal)

except StopIteration:
    print("\n✅ ダウンロード完了（全race_id処理済み）")

print(f"\n📊 最終統計:")
print(f"   処理件数: {total_processed}件")
print(f"   新規ダウンロード: {stats['success']}件")
print(f"   既存スキップ(ディスクに存在): {stats['exists']}件")
print(f"   未発見: {stats['not_found']}件")
print(f"   エラー: {stats['error']}件")
if last_success_race:
    print(f"   最新レース: {last_success_race}")
